# MCTS Eval (Colab) — pillar2x variants

Compare pillar2x training runs on 50 seeds at 400 sims.
Run on G4/L4/A100 for fast GPU inference.

**Critical:** BATCH_SIZE=8 (HISTORY lesson 115 — virtual-loss starvation
kills MCTS quality at bs > 8).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `feature_value_weights.npz` — feature-value evaluator (policy_only models need it)
3. The pillar2x checkpoints to compare

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy feature-value evaluator (required for policy_only models)
os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/feature_value_weights.npz /content/alphatrain/data/feature_value_weights.npz
print(f'Feature weights: {os.path.getsize("/content/alphatrain/data/feature_value_weights.npz")} bytes')

!pip install -q numpy numba scipy

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === EDIT THESE ===
# pillar2x  (lr=1e-4): the original V10 training run
# pillar2x2 (lr=3e-4): the parallel run with higher lr
CHECKPOINTS = [
    ('2X',  f'{DRIVE}/pillar2x_epoch_6.pt'),
    ('2X2', f'{DRIVE}/pillar2x2_epoch_6.pt'),
]
SIMS = 400
SEEDS = ' '.join(str(i) for i in range(50))   # 0..49
WORKERS = 24
BATCH_SIZE = 8         # MCTS quality requires bs=8 (HISTORY lesson 115)
FEATURE_WEIGHTS = '/content/alphatrain/data/feature_value_weights.npz'
# ==================

for label, path in CHECKPOINTS:
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'OK: {label} ({os.path.getsize(path)/1e6:.0f} MB)')

In [ ]:
%cd /content

for label, path in CHECKPOINTS:
    print(f'\n{"="*60}')
    print(f'=== {label}: {path.split("/")[-1]} ===')
    print(f'{"="*60}\n', flush=True)
    !python -m alphatrain.scripts.eval_parallel \
        --model {path} \
        --feature-value-weights {FEATURE_WEIGHTS} \
        --device cuda --workers {WORKERS} --batch-size {BATCH_SIZE} \
        --simulations {SIMS} --games-per-seed 1 \
        --early-stop --compile \
        --seeds {SEEDS}